# Eclipse stack pipeline (v0)

This notebook shows **what each stage does** (loops, merges, patch grids, where FFTs run). Heavy math stays in `eclipse_v0/*.py`; here you see the **control flow** and parameters.

Artifacts use the `v0-` prefix under `WORKDIR`.

In [ ]:
import sys
from pathlib import Path


def v0_package_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "eclipse_v0").is_dir():
        return cwd
    if (cwd / "v0" / "eclipse_v0").is_dir():
        return cwd / "v0"
    raise RuntimeError(
        "Could not find eclipse_v0: set cwd to the `v0` directory or the repo root."
    )


V0_ROOT = v0_package_root()
if str(V0_ROOT) not in sys.path:
    sys.path.insert(0, str(V0_ROOT))

In [ ]:
# GPU selection — same defaults as legacy `eda00.py`. Override in the shell, e.g.
#   CUDA_VISIBLE_DEVICES=0 jupyter lab
from eclipse_v0.device import configure_cuda_visible_devices, require_cuda

configure_cuda_visible_devices()
require_cuda()

In [2]:
import os
from pathlib import Path

DATA_ROOT = Path(os.environ.get("EDA00_DATA_ROOT", "/home/slavik/e202602_eclipse/data"))
WORKDIR = Path(os.environ.get("ECLIPSE_V0_WORKDIR", "/home/slavik/tmp/eclipse_v0_run"))
WORKDIR.mkdir(parents=True, exist_ok=True)

PK_EDA00 = WORKDIR / "v0-eda00.pkl"
PK_EDA02 = WORKDIR / "v0-eda02.pkl"
PK_EDA03 = WORKDIR / "v0-eda03.pkl"

## Stage 0 — ingest, moon, intra-exposure registration

1. **Scan** JPEGs under `DATA_ROOT` (brightness filter, EXIF).
2. **Per image (GPU loop):** coarse disk finder → gradient/triplet **moon** refine.
3. **Group** by exposure time; print stats.
4. **Radius gate:** drop inconsistent moon radii across the rolling reference.
5. **Interpolate** missing `(i,j)` from a time-linear fit; fill radii from neighbors.
6. **Uncertainty:** `moon_pos_std_px` from residuals vs that fit.
7. **Per exposure, every ordered pair `(i,j)`:** Fourier-style registration on GPU; closure checks (pair + triplet).
8. **Pickle** `exposure_groups` + `reg`.

### 0a — Scan disk → `ImageInfo` list

In [ ]:
from eclipse_v0 import stage0 as s0

image_infos = s0.get_image_infos(DATA_ROOT)
len(image_infos), image_infos[0].path.name

### 0b — Moon detection (loop over every frame)

In [ ]:
s0.stage0_detect_moons(image_infos)

### 0c — Group by exposure; drop radius outliers

In [ ]:
exposure_groups = s0.stage0_group_by_exposure(image_infos)
s0.stage0_prune_radius_outliers(exposure_groups)

### 0d — Interpolate missing positions; set position uncertainty

In [ ]:
interp = s0.stage0_interpolate_missing_moons(image_infos, exposure_groups)
s0.stage0_set_moon_position_std(image_infos, exposure_groups, interp)

### 0e — Intra-exposure registration: all ordered pairs per group (slow)

In [ ]:
reg = s0.stage0_register_intra_exposure_pairs(exposure_groups)
s0.stage0_save_pickle(exposure_groups, reg, PK_EDA00)

## Stage 2 — prune stacks, global pose fit per exposure

1. **Load** stage-0 pickle.
2. **Prune:** brightness outliers (large groups), then triplet-inconsistent frames.
3. **Per exposure (loop):** treat pairwise `reg` as observations; **Adam** first on angles then shifts so implied transforms match all edges (two 50k phases in `run_group`).
4. **Debug outputs:** random crop, weighted-mean crop, GIF of aligned frames per exposure.
5. **Pickle** pruned groups, plain `reg`, and `opt_results` (`abs_xy`, `abs_angle_t`).

In [ ]:
import torch
from eclipse_v0 import stage2 as s2

exposure_groups, reg = s2.stage2_load(PK_EDA00)
s2.stage2_prune_groups(exposure_groups, reg)

### 2b — Optimization loop (one solve per exposure time)

In [ ]:
device = torch.device("cuda")
opt_results = s2.stage2_optimize_poses_and_debug(
    exposure_groups, reg, device, debug_img_dir=WORKDIR
)

In [ ]:
s2.stage2_save_pickle(PK_EDA02, exposure_groups, reg, opt_results)

## Stage 3 — full-res stack mean per exposure, then cross-exposure chain

1. **Load** stage-2 pickle.
2. **Moon table:** median `(i,j,r)` per exposure (masks for gamma / registration).
3. **Loop exposures (time order):** rebuild **full-size weighted average** (same masking/warp as stage 2).
4. **Loop consecutive pairs `(t0,t1)`:** estimate **gamma** (MAE outside moon); **register** longer stack to shorter using masked Fourier metric (implementation in library); **refine gamma** after alignment; save comparison GIF.
5. **Pickle** `cross_reg` and `gamma_by_pair`.

In [ ]:
from eclipse_v0 import stage3 as s3

exposure_groups, _reg, opt_results = s3.stage3_load(PK_EDA02)
moon_by_exp, exposure_times_sorted = s3.stage3_moon_median_table(exposure_groups)

### 3b — Full-resolution averaged image per exposure

In [ ]:
avg_images = s3.stage3_fullsize_averages(
    exposure_groups, exposure_times_sorted, opt_results, device
)

### 3c — Consecutive exposure pairs: gamma + cross registration

In [ ]:
pairs_results = s3.stage3_cross_exposure_consecutive_pairs(
    exposure_times_sorted, avg_images, moon_by_exp, device, pair_gif_dir=WORKDIR
)
s3.stage3_save_pickle(PK_EDA03, pairs_results)

## Stage 5 — reference merge, radial tone, **patch FFT sharpen**, RGB

The next code cell uses a **`Stage5Context`** and explicit **`stage5_*` steps** (same math as `run_stage5(WORKDIR)`).

**A →** `stage5_load_inputs` — reload `v0-eda02.pkl` / `v0-eda03.pkl`; drop first two exposure times; set reference and `moon_ref`.

**B →** `stage5_build_per_exposure_averages` then `stage5_warp_merge_to_composite` — GPU stack means; chain `cross_reg` + gamma scaling; weighted merge to reference grid.

**C →** `stage5_crop_and_save_composite` — mutual-valid crop; `v0-eda05_composite.npy` + preview.

**D →** `stage5_radial_normalize_display` — moon refine, polar tone + **p3** stretch → `ctx.display`.

**E →** `stage5_fft_unsharp_and_save` — patch FFT unsharp (σ ∈ {2,4,8}, weights `STAGE5_UNSHARP_WEIGHTS`).

**F →** `stage5_rgb_vignette_and_radial_pickle` — RGB + vignette; `v0-eda05_radial.pkl`.

**A. Reload** stage 2+3 pickles; drop first two exposure times (same as `eda05`); pick **reference** = shortest remaining; recompute per-exposure averages in that time list.

**B. Warp + weighted merge** every exposure into the reference grid (chain `cross_reg`, gamma-based intensity scaling, radial weights).

**C. Crop** to mutual valid footprint; save float composite + preview PNG.

**D. Radial pipeline (GPU):** warp to polar, angular statistics / extrapolation, piecewise tone map, **p3** percentile curve, normalize — details in library; outcome is a display-range grayscale `display`.

**E. FFT sharpen (this is the patch loop you asked to surface):**
- Build **difference** `display - blurred(display)` with plain Gaussian and a polar-band blur (library).
- **Tile** the image with square patches (`STAGE5_PATCH_SIDE` × `STAGE5_PATCH_SIDE`, stride `STAGE5_PATCH_STRIDE`).
- **For each patch origin** `(r0, c0)` and each blur scale **σ ∈ {2,4,8}**:
  - `FFT2` → `fftshift` → **spectral processing** (amplitude shaping in polar layout + percentile gates; moon-aware masking — **hidden in library**) → `ifftshift` → `IFFT2` → real part.
  - **Overlap-add** with a smooth circular window.
- **Combine** the three smoothed residual maps with weights `STAGE5_UNSHARP_WEIGHTS`, add back to `display`, clip, zero moon disk.

**F. RGB + vignette** on sharpened gray; save PNG and `v0-eda05_radial.pkl` sidecar.

In [3]:
import pickle
from eclipse_v0.stage5 import (
    STAGE5_FFT_INWARD_MEDIAN_SPAN,
    STAGE5_PATCH_SIDE,
    STAGE5_PATCH_STRIDE,
    STAGE5_UNSHARP_GAUSSIAN_SIGMAS,
    STAGE5_UNSHARP_WEIGHTS,
    stage5_count_fft_patch_placements,
)

with open(PK_EDA02, "rb") as fd:
    _eg = pickle.load(fd)
_sample = next(iter(_eg.values()))[0]
H0, W0 = _sample.height, _sample.width
nr, nc, n_tot = stage5_count_fft_patch_placements(H0, W0)
print(
    "FFT sharpen patch scan (on radial output, size ≈ crop — often smaller than full frame):\n"
    f"  full-frame geometry if we ran on {H0}x{W0}: grid {nr}x{nc} origins = {n_tot} patches per σ\n"
    f"  patch={STAGE5_PATCH_SIDE}px stride={STAGE5_PATCH_STRIDE}\n"
    f"  Gaussian σ for residual blur: {STAGE5_UNSHARP_GAUSSIAN_SIGMAS}\n"
    f"  combine weights: {STAGE5_UNSHARP_WEIGHTS}\n"
    f"  polar median span on |F| (library): {STAGE5_FFT_INWARD_MEDIAN_SPAN}"
)

FFT sharpen patch scan (on radial output, size ≈ crop — often smaller than full frame):
  full-frame geometry if we ran on 4000x6000: grid 469x719 origins = 337211 patches per σ
  patch=256px stride=8
  Gaussian σ for residual blur: (2.0, 4.0, 8.0)
  combine weights: (8.0, 8.0, 0.5)
  polar median span on |F| (library): 32


In [4]:
from eclipse_v0.stage5 import (
    Stage5Context,
    stage5_build_per_exposure_averages,
    stage5_crop_and_save_composite,
    stage5_fft_unsharp_and_save,
    stage5_load_inputs,
    stage5_radial_normalize_display,
    stage5_rgb_vignette_and_radial_pickle,
    stage5_warp_merge_to_composite,
)

ctx = Stage5Context(workdir=WORKDIR)
stage5_load_inputs(ctx)
stage5_build_per_exposure_averages(ctx)
stage5_warp_merge_to_composite(ctx)
stage5_crop_and_save_composite(ctx)
stage5_radial_normalize_display(ctx)
stage5_fft_unsharp_and_save(ctx)
stage5_rgb_vignette_and_radial_pickle(ctx)

Reference exposure t_ref=0.001, moon_ref (i,j,r)=(1965.4510573255507, 2884.309810373511, 316.0109492675125)
cross_reg pairs: 16, gamma_by_pair: 16


Averaged images: 100%|██████████| 15/15 [00:24<00:00,  1.63s/it]


Built 15 averaged images.
Reference shape H=4000, W=6000
Exposures in chain: 15


Warp and merge: 15it [00:06,  2.26it/s]


Composite shape (4000, 6000), dtype float64
Crop bounds rows [21,3979], cols [17,5973]; shape (3959, 5957)
Saved /home/slavik/tmp/eclipse_v0_run/v0-eda05_composite.npy (float64), /home/slavik/tmp/eclipse_v0_run/v0-eda05_composite_preview.png


100%|██████████| 3490/3490 [00:01<00:00, 2628.72it/s]


Saved /home/slavik/tmp/eclipse_v0_run/v0-eda05_radial_normalize.png


FFT-smooth diff sigma=8.0: 100%|██████████| 463/463 [04:25<00:00,  1.74it/s]


Saved /home/slavik/tmp/eclipse_v0_run/v0-eda05_radial_normalize_sharpen_fft_smoothed_diff.png
Saved /home/slavik/tmp/eclipse_v0_run/v0-eda05_rgb_rescaled.png
Saved /home/slavik/tmp/eclipse_v0_run/v0-eda05_radial.pkl


### After run: patch count on actual composite crop

In [ ]:
import numpy as np
from eclipse_v0.stage5 import stage5_count_fft_patch_placements

comp = np.load(WORKDIR / "v0-eda05_composite.npy")
h, w = comp.shape
nr, nc, nt = stage5_count_fft_patch_placements(h, w)
print(f"Composite crop {h}x{w}: FFT sharpen uses {nr}x{nc} = {nt} patch starts per σ")

## Quick look (optional)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

composite = np.load(WORKDIR / "v0-eda05_composite.npy")
rgb = np.asarray(Image.open(WORKDIR / "v0-eda05_rgb_rescaled.png")) / 255.0
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(composite, cmap="gray")
axes[0].set_title("composite (float crop)")
axes[1].imshow(rgb)
axes[1].set_title("RGB export")
plt.tight_layout()
plt.show()

---

**One-shot alternative** (same math, fewer visible steps): `s0.run_stage0(DATA_ROOT, PK_EDA00)`, `s2.run_stage2(...)`, `s3.run_stage3(...)`, and either `run_stage5(WORKDIR)` or the stepped `Stage5Context` + `stage5_*` calls above.